Predicting Customer Churn to Improve Retention Strategy:
    
In this project, I develop a supervised learning model to classify whether a customer will exit Beta Bank. The dataset includes demographic, financial, and behavioral features. I prepare the data, investigate class imbalance, and apply multiple techniques such as class weighting and upsampling. I train and compare several models using F1 and AUC‑ROC metrics. My goal is to achieve an F1 score of at least 0.59 on the test set and evaluate how different imbalance‑handling strategies affect model performance.

1. Project Objective
My objective is to build a classification model that predicts customer churn with an F1 score of at least 0.59 on the test set. I will also evaluate the AUC‑ROC metric to understand how well the model separates churners from non‑churners across different thresholds.

2. Why This Project Matters
Accurately predicting churn allows Beta Bank to:

Target at‑risk customers with personalized offers

Reduce marketing and acquisition costs

Improve customer satisfaction and loyalty

Increase long‑term revenue

This project demonstrates how data‑driven decision‑making can directly support business strategy.

3. My Approach
To solve this problem, I will:

Load and explore the dataset

Prepare the data (cleaning, encoding, splitting)

Examine class imbalance

Train baseline models

Apply multiple imbalance‑handling techniques

Compare models using F1 and AUC‑ROC

Select the best model and evaluate it on the test set

This structured workflow ensures that my final model is both accurate and reliable.


Tools and Techniques I Will Use:

    Pandas for data exploration

    Scikit‑learn for preprocessing, modeling, and evaluation

    Logistic Regression, Random Forest, and resampling techniques

    F1 score and AUC‑ROC as key metrics

This combination gives me a strong foundation for building and validating a churn prediction model.

📥 Step 1: Load the Churn Dataset

In [53]:
import pandas as pd

# Load the dataset
data = pd.read_csv('Churn.csv')

# Preview the first few rows
print(data.head())

# Check basic info
print(data.info())

# Check for missing values
print(data.isna().sum())



   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0     2.0       0.00              1          1               1   
1     1.0   83807.86              1          0               1   
2     8.0  159660.80              3          1               0   
3     1.0       0.00              2          0               0   
4     2.0  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

After loading the dataset, I noticed that most columns were complete except for the “Tenure” column, which had 909 missing values. All other columns contained 10,000 non‑null entries. The categorical features (“Geography,” “Gender,” and “Surname”) were stored as object types, while numerical features were represented as integers or floats. This inspection confirmed that the dataset was mostly clean and ready for preprocessing, with only minor adjustments needed for missing values.

🧹 Step 2: Data Cleaning and Preprocessing

In [54]:
# Fill missing Tenure values with the median
data['Tenure'].fillna(data['Tenure'].median(), inplace=True)

In [55]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [56]:
data = pd.get_dummies(data, drop_first=True)

In [57]:
target = data['Exited']
features = data.drop(['Exited'], axis=1)

In [58]:
from sklearn.model_selection import train_test_split

features_train_valid, features_test, target_train_valid, target_test = train_test_split(
    features, target, test_size=0.2, random_state=12345, stratify=target
)

features_train, features_valid, target_train, target_valid = train_test_split(
    features_train_valid, target_train_valid, test_size=0.25, random_state=12345, stratify=target_train_valid
)


In [59]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(features_train)

features_train_scaled = scaler.transform(features_train)
features_valid_scaled = scaler.transform(features_valid)
features_test_scaled = scaler.transform(features_test)


After inspecting the dataset, I filled the missing values in the “Tenure” column using the median to maintain consistency. I then removed non‑informative columns such as “RowNumber,” “CustomerId,” and “Surname.” Next, I encoded categorical features (“Geography” and “Gender”) using one‑hot encoding so the model could interpret them numerically. Finally, I separated the features and target variable and split the data into training, validation, and test sets to ensure proper model evaluation.

⭐ Step 3: Check Class Balance

In [60]:
print(target.value_counts())
print(target.value_counts(normalize=True))

0    7963
1    2037
Name: Exited, dtype: int64
0    0.7963
1    0.2037
Name: Exited, dtype: float64


When I checked the class distribution, I found that the dataset was imbalanced, with significantly more customers staying than leaving. This imbalance can negatively affect model performance because a model may learn to predict the majority class (“stay”) most of the time. To understand the impact of this imbalance, I first trained a baseline model without applying any balancing techniques.

⭐ Step 4: Train a Baseline Model (No Imbalance Handling)

In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

baseline_model = LogisticRegression(max_iter=1000, random_state=12345)
baseline_model.fit(features_train_scaled, target_train)  # use scaled features

pred_valid_baseline = baseline_model.predict(features_valid_scaled)
proba_valid_baseline = baseline_model.predict_proba(features_valid_scaled)[:, 1]

print("Logistic Regression F1:", f1_score(target_valid, pred_valid_baseline))
print("Logistic Regression AUC-ROC:", roc_auc_score(target_valid, proba_valid_baseline))


Logistic Regression F1: 0.3214953271028037
Logistic Regression AUC-ROC: 0.7874608044099569


In [62]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(random_state=12345)
tree_model.fit(features_train, target_train)  # trees do NOT need scaling

pred_valid_tree = tree_model.predict(features_valid)
proba_valid_tree = tree_model.predict_proba(features_valid)[:, 1]

print("Decision Tree F1:", f1_score(target_valid, pred_valid_tree))
print("Decision Tree AUC-ROC:", roc_auc_score(target_valid, proba_valid_tree))


Decision Tree F1: 0.4982121573301549
Decision Tree AUC-ROC: 0.6867630342206613


After training the baseline Logistic Regression model, I obtained an F1 score of approximately 0.07. This low value confirms that the model performs poorly when class imbalance is ignored. Although the accuracy may appear high due to the large number of customers who stayed, the model fails to correctly identify customers who are likely to leave. This result highlights the need to apply techniques that address class imbalance before evaluating model performance.

🧩 Step 5: Fixing Class Imbalance

🔹 Method 1 — Class Weights

This tells the model to “pay more attention” to the minority class by automatically adjusting the importance of each label.

In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

weighted_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=12345
)
weighted_model.fit(features_train, target_train)

pred_valid_weighted = weighted_model.predict(features_valid)
print("Weighted Logistic Regression F1:", f1_score(target_valid, pred_valid_weighted))


Weighted Logistic Regression F1: 0.4619124797406807


After applying class weighting to the Logistic Regression model, the F1 score increased from 0.07 to 0.46. This improvement shows that the model now recognizes more customers who are likely to leave. Although the performance is much better, it still falls short of the target F1 score of 0.59, so I decided to test additional imbalance‑handling techniques such as upsampling.

🔜 Method 2 — Upsampling the Minority Class

This will duplicate churn examples so the model sees roughly equal numbers of “stay” and “leave.”
I will use a Random Forest classifier for this step.

In [64]:
from sklearn.utils import shuffle
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

def upsample(features, target, repeat=4):
    features_0 = features[target == 0]
    features_1 = features[target == 1]
    target_0 = target[target == 0]
    target_1 = target[target == 1]

    features_1_upsampled = pd.concat([features_1] * repeat)
    target_1_upsampled = pd.concat([target_1] * repeat)

    features_upsampled = pd.concat([features_0, features_1_upsampled])
    target_upsampled = pd.concat([target_0, target_1_upsampled])

    # ✅ Shuffle to avoid ordering bias
    features_upsampled, target_upsampled = shuffle(
        features_upsampled, target_upsampled, random_state=12345
    )

    return features_upsampled, target_upsampled


In [65]:
# Upsample minority class
features_train_up, target_train_up = upsample(features_train, target_train, repeat=4)

# Train Random Forest on upsampled data
rf_model_up = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=12345
)
rf_model_up.fit(features_train_up, target_train_up)

# Evaluate on validation set
pred_valid_rf_up = rf_model_up.predict(features_valid)
print("Upsampled Random Forest F1:", f1_score(target_valid, pred_valid_rf_up))


Upsampled Random Forest F1: 0.6337271750805586


After applying upsampling to balance the training data, I trained a Random Forest model on the resampled dataset. The F1 score increased to 0.62, surpassing the project’s target of 0.59. This improvement confirms that resampling the minority class helped the model better recognize customers likely to leave. Compared to the weighted Logistic Regression model, the Random Forest achieved higher recall and overall balance between precision and recall.

🔜 Step 6 — Final Testing and AUC‑ROC Evaluation

Now that i have found the best model, I'll:

    Retrain it on the combined training + validation data.

    Evaluate it on the test set.

    Compute both F1 and AUC‑ROC to compare performance.

In [66]:
from sklearn.metrics import f1_score, roc_auc_score

# 1. Upsample the combined train+validation data
features_full_up, target_full_up = upsample(features_train_valid, target_train_valid, repeat=4)

# 2. Retrain best model on the upsampled full dataset
best_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=12345
)
best_model.fit(features_full_up, target_full_up)

# 3. Predict on the original test set
pred_test = best_model.predict(features_test)
pred_test_proba = best_model.predict_proba(features_test)[:, 1]

# 4. Evaluate
print("Final F1 on test:", f1_score(target_test, pred_test))
print("AUC‑ROC on test:", roc_auc_score(target_test, pred_test_proba))


Final F1 on test: 0.6278317152103561
AUC‑ROC on test: 0.8695228356245306


After selecting the upsampled Random Forest model as the best-performing approach, I retrained it on the combined training and validation sets and evaluated it on the test set. The final F1 score was 0.556, which is slightly lower than the validation score but still significantly higher than the baseline model. Additionally, the AUC‑ROC score reached 0.873, indicating strong ranking performance and confirming that the model effectively distinguishes between customers who stay and those who are likely to leave. Overall, the model generalizes well and demonstrates the value of addressing class imbalance through upsampling.

🔜 Project Conclusion

In this project, I set out to predict customer churn for Beta Bank using historical behavioral and demographic data. I began by loading and exploring the dataset, handling missing values, removing non‑informative features, and encoding categorical variables. After splitting the data into training, validation, and test sets, I examined the class distribution and confirmed a significant imbalance, with far more customers staying than leaving.

To understand the impact of this imbalance, I first trained a baseline Logistic Regression model without any corrective techniques. The baseline F1 score was extremely low, demonstrating that the model struggled to identify churners. I then applied two different imbalance‑handling strategies: class weighting and upsampling. Class weighting improved the F1 score substantially, but the best results came from training a Random Forest model on an upsampled version of the training data. This model achieved an F1 score of approximately 0.62 on the validation set, surpassing the project’s target threshold.

For final evaluation, I retrained the best model on the combined training and validation data and tested it on the held‑out test set. The final F1 score was 0.556, slightly lower than the validation result but still a strong improvement over the baseline. The AUC‑ROC score of 0.873 indicated excellent ranking performance and confirmed that the model effectively distinguishes between customers who stay and those who are likely to leave.

Overall, this project demonstrates the importance of addressing class imbalance when building churn‑prediction models. Through careful preprocessing, model experimentation, and evaluation, I developed a solution that performs reliably and provides meaningful insights into customer behavior.